# PetShop Huellitas — Auditoría y preparación de datos

Este notebook implementa un flujo reproducible de auditoría, limpieza inicial, validación e integridad referencial.

Principios del proceso:

- Los archivos de `datos/raw` nunca se modifican.
- Todas las transformaciones se realizan sobre copias.
- Los registros problemáticos se separan para revisión; no desaparecen silenciosamente.
- Las validaciones finales comprueban que el proceso pueda ejecutarse desde un kernel limpio.

Dependencias del entorno: `pandas`, `numpy`, `openpyxl`, `jupyter` e `ipykernel`.


## 1. Configuración y carga


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

DIRECTORIO_ACTUAL = Path.cwd().resolve()
RUTA_PROYECTO = (
    DIRECTORIO_ACTUAL.parent
    if DIRECTORIO_ACTUAL.name.lower() == "notebooks"
    else DIRECTORIO_ACTUAL
)
RUTA_DATOS = RUTA_PROYECTO / "datos" / "raw"
RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"

ARCHIVOS_REQUERIDOS = [
    "ventas.csv",
    "productos.xlsx",
    "clientes.xlsx",
    "stock.csv",
]

archivos_faltantes = [
    nombre for nombre in ARCHIVOS_REQUERIDOS
    if not (RUTA_DATOS / nombre).is_file()
]

print("Directorio del proyecto:", RUTA_PROYECTO)
print("Directorio de datos:", RUTA_DATOS)

if archivos_faltantes:
    raise FileNotFoundError(
        "Faltan archivos en datos/raw: " + ", ".join(archivos_faltantes)
    )

print("✓ Los cuatro archivos requeridos están disponibles.")


Directorio del proyecto: /home/ips/Repositorio/Análisis de Datos/huellitas
Directorio de datos: /home/ips/Repositorio/Análisis de Datos/huellitas/datos/raw
✓ Los cuatro archivos requeridos están disponibles.


In [2]:
ventas = pd.read_csv(
    RUTA_DATOS / "ventas.csv",
    encoding="utf-8-sig",
    dtype={
        "id_venta": "string",
        "id_cliente": "string",
        "id_producto": "string",
    },
)

productos = pd.read_excel(
    RUTA_DATOS / "productos.xlsx",
    dtype={"id_producto": "string"},
)

clientes = pd.read_excel(
    RUTA_DATOS / "clientes.xlsx",
    dtype={"id_cliente": "string"},
)

stock = pd.read_csv(
    RUTA_DATOS / "stock.csv",
    encoding="utf-8-sig",
    dtype={"id_producto": "string"},
)

datasets_originales = {
    "ventas": ventas,
    "productos": productos,
    "clientes": clientes,
    "stock": stock,
}

print("✓ Archivos cargados correctamente.")


✓ Archivos cargados correctamente.


## 2. Auditoría inicial


In [3]:
resumen_calidad = pd.DataFrame([
    {
        "archivo": nombre,
        "filas": len(df),
        "columnas": len(df.columns),
        "filas_duplicadas": int(df.duplicated().sum()),
        "celdas_vacias": int(df.isna().sum().sum()),
        "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 3),
    }
    for nombre, df in datasets_originales.items()
])

display(resumen_calidad)

for nombre, df in datasets_originales.items():
    print(f"\n{nombre.upper()}: {df.shape[0]} filas × {df.shape[1]} columnas")
    display(df.head())


,archivo,filas,columnas,filas_duplicadas,celdas_vacias,memoria_mb
0,ventas,20025,10,25,2759,8.130
1,productos,200,8,0,8,0.080
2,clientes,2010,6,10,38,0.741
3,stock,607,5,7,6,0.108



VENTAS: 20025 filas × 10 columnas


,id_venta,fecha,id_cliente,id_producto,cantidad,precio_unitario,descuento,sucursal,canal,medio_pago
0,V000001,17/03/2025,C0158,P0167,1,55310,0.20,Ecommerce,Online,Transferencia
1,V000002,04/07/2026,C0706,P0022,1,6310,0.00,Centro,Local,Transferencia
2,V000002,04/07/2026,C0706,P0035,1,8930,0.00,Centro,Local,Transferencia
3,V000003,03/03/2025,<NA>,P0003,1,24130,0.00,Ecommerce,Online,Transferencia
4,V000003,03/03/2025,<NA>,P0118,1,11040,0.05,Ecommerce,Online,Transferencia



PRODUCTOS: 200 filas × 8 columnas


,id_producto,producto,categoria,subcategoria,marca,costo,precio_lista,proveedor
0,P0001,Pedigree Cachorro Perro 3 kg,Alimentos,Perro,Pedigree,15300.0,23900,Mundo Mascota
1,P0002,"König Piel sensible Gato 7,5 kg",Alimentos,Gato,König,4500.0,7400,Pet Supply BA
2,P0003,Excellent Premium Perro 10 kg,Alimentos,Perro,Excellent,13800.0,25300,Mayorista Animalia
3,P0004,Balanced Cachorro Gato 15 kg,Alimentos,Gato,Balanced,14100.0,26300,Veterinaria Central
4,P0005,Catit Piel sensible Perro 1 kg,Alimentos,Perro,Catit,13600.0,22300,EcoPet Distribuciones



CLIENTES: 2010 filas × 6 columnas


,id_cliente,nombre,ciudad,fecha_alta,mascota,email
0,C0001,Santiago Suárez,City Bell,2024-02-16 00:00:00,Gato,santiago.suarez.1@hotmail.com
1,C0002,Franco Álvarez,Gonnet,2026-05-18 00:00:00,Perro,franco.alvarez.2@gmail.com
2,C0003,Franco Torres,City Bell,2023-11-19 00:00:00,Gato,franco.torres.3@gmail.com
3,C0004,Agustina Herrera,La Plata,2025-12-26 00:00:00,Gato,agustina.herrera.4@gmail.com
4,C0005,Bautista Benítez,NaN,2023-11-27 00:00:00,Gato,bautista.benitez.5@yahoo.com.ar



STOCK: 607 filas × 5 columnas


,fecha,sucursal,id_producto,stock_actual,stock_minimo
0,31/08/2026,Centro,P0001,26,13.0
1,31/08/2026,Centro,P0002,7,7.0
2,31/08/2026,Centro,P0003,3,7.0
3,31/08/2026,Centro,P0004,11,6.0
4,31/08/2026,centro,P0005,7,5.0


In [4]:
resumen_faltantes = []

for nombre, df in datasets_originales.items():
    for columna in df.columns:
        cantidad = int(df[columna].isna().sum())
        if cantidad > 0:
            resumen_faltantes.append({
                "archivo": nombre,
                "columna": columna,
                "cantidad": cantidad,
                "porcentaje": round(cantidad / len(df) * 100, 2),
            })

resumen_faltantes = pd.DataFrame(resumen_faltantes)
display(resumen_faltantes)


,archivo,columna,cantidad,porcentaje
0,ventas,fecha,1,0.00
1,ventas,id_cliente,2758,13.77
2,productos,costo,5,2.50
3,productos,proveedor,3,1.50
4,clientes,ciudad,33,1.64
5,clientes,email,5,0.25
6,stock,stock_minimo,6,0.99


## 3. Copias de trabajo y duplicados

Una repetición de `id_venta` no implica un duplicado: una misma venta puede contener varios productos. En esta etapa solo se eliminan filas idénticas en todas sus columnas.


In [5]:
ventas_limpias = ventas.copy()
productos_limpios = productos.copy()
clientes_limpios = clientes.copy()
stock_limpio = stock.copy()

registro_limpieza = []


def eliminar_duplicados_exactos(df, nombre_tabla):
    filas_antes = len(df)
    resultado = df.drop_duplicates().reset_index(drop=True)
    filas_despues = len(resultado)

    registro_limpieza.append({
        "tabla": nombre_tabla,
        "proceso": "Eliminación de duplicados exactos",
        "filas_antes": filas_antes,
        "filas_despues": filas_despues,
        "filas_afectadas": filas_antes - filas_despues,
        "criterio": "Coincidencia en todas las columnas",
    })
    return resultado


ventas_limpias = eliminar_duplicados_exactos(ventas_limpias, "ventas")
productos_limpios = eliminar_duplicados_exactos(productos_limpios, "productos")
clientes_limpios = eliminar_duplicados_exactos(clientes_limpios, "clientes")
stock_limpio = eliminar_duplicados_exactos(stock_limpio, "stock")

registro_limpieza_df = pd.DataFrame(registro_limpieza)
display(registro_limpieza_df)

for nombre, df in {
    "ventas": ventas_limpias,
    "productos": productos_limpios,
    "clientes": clientes_limpios,
    "stock": stock_limpio,
}.items():
    assert df.duplicated().sum() == 0, f"Quedan duplicados exactos en {nombre}"

print("✓ No quedan duplicados exactos.")


,tabla,proceso,filas_antes,filas_despues,filas_afectadas,criterio
0,ventas,Eliminación de duplicados exactos,20025,20000,25,Coincidencia en todas las columnas
1,productos,Eliminación de duplicados exactos,200,200,0,Coincidencia en todas las columnas
2,clientes,Eliminación de duplicados exactos,2010,2000,10,Coincidencia en todas las columnas
3,stock,Eliminación de duplicados exactos,607,600,7,Coincidencia en todas las columnas


✓ No quedan duplicados exactos.


In [6]:
duplicados_clave_productos = productos_limpios[
    productos_limpios.duplicated("id_producto", keep=False)
]
duplicados_clave_clientes = clientes_limpios[
    clientes_limpios.duplicated("id_cliente", keep=False)
]

assert duplicados_clave_productos.empty, "Hay id_producto repetidos"
assert duplicados_clave_clientes.empty, "Hay id_cliente repetidos"

if "id_linea_venta" not in ventas_limpias.columns:
    ventas_limpias.insert(
        0,
        "id_linea_venta",
        [f"LV{numero:06d}" for numero in range(1, len(ventas_limpias) + 1)],
    )

assert ventas_limpias["id_linea_venta"].is_unique
print("✓ Claves maestras válidas e identificador de línea creado.")


✓ Claves maestras válidas e identificador de línea creado.


## 4. Normalización de textos


In [7]:
bitacora_transformaciones = []


def normalizar_espacios(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def registrar_transformacion(tabla, columna, proceso, antes, despues):
    cambios = int(
        (antes.fillna("<VACIO>") != despues.fillna("<VACIO>")).sum()
    )
    bitacora_transformaciones.append({
        "tabla": tabla,
        "columna": columna,
        "proceso": proceso,
        "registros_modificados": cambios,
    })


columnas_identificadores = {
    "ventas": (ventas_limpias, ["id_linea_venta", "id_venta", "id_cliente", "id_producto"]),
    "productos": (productos_limpios, ["id_producto"]),
    "clientes": (clientes_limpios, ["id_cliente"]),
    "stock": (stock_limpio, ["id_producto"]),
}

for tabla, (df, columnas) in columnas_identificadores.items():
    for columna in columnas:
        antes = df[columna].copy()
        df[columna] = normalizar_espacios(df[columna]).str.upper()
        registrar_transformacion(
            tabla, columna,
            "Eliminación de espacios y conversión a mayúsculas",
            antes, df[columna],
        )


In [8]:
mapa_sucursales = {
    "centro": "Centro",
    "sucursal centro": "Centro",
    "city bell": "City Bell",
    "sucursal city bell": "City Bell",
    "los hornos": "Los Hornos",
    "sucursal los hornos": "Los Hornos",
    "ecommerce": "Ecommerce",
}

mapa_categorias = {
    "alimento": "Alimentos",
    "alimentos": "Alimentos",
    "snack": "Snacks",
    "snacks": "Snacks",
    "higiene": "Higiene",
    "accesorio": "Accesorios",
    "accesorios": "Accesorios",
    "salud": "Salud",
}

mapa_ciudades = {
    "la plata": "La Plata",
    "lp": "La Plata",
    "l.p.": "La Plata",
    "city bell": "City Bell",
    "los hornos": "Los Hornos",
    "gonnet": "Gonnet",
    "villa elisa": "Villa Elisa",
    "berisso": "Berisso",
    "ensenada": "Ensenada",
    "tolosa": "Tolosa",
    "ringuelet": "Ringuelet",
}


def aplicar_mapa(df, tabla, columna, mapa, proceso):
    antes = df[columna].copy()
    clave = normalizar_espacios(antes).str.casefold()
    df[columna] = clave.map(mapa).fillna(normalizar_espacios(antes))
    registrar_transformacion(tabla, columna, proceso, antes, df[columna])


aplicar_mapa(ventas_limpias, "ventas", "sucursal", mapa_sucursales, "Unificación de sucursales")
aplicar_mapa(stock_limpio, "stock", "sucursal", mapa_sucursales, "Unificación de sucursales")
aplicar_mapa(productos_limpios, "productos", "categoria", mapa_categorias, "Unificación de categorías")
aplicar_mapa(clientes_limpios, "clientes", "ciudad", mapa_ciudades, "Unificación de ciudades")

for tabla, df, columnas in [
    ("productos", productos_limpios, ["producto", "subcategoria", "marca", "proveedor"]),
    ("clientes", clientes_limpios, ["nombre", "mascota"]),
    ("ventas", ventas_limpias, ["canal", "medio_pago"]),
]:
    for columna in columnas:
        antes = df[columna].copy()
        df[columna] = normalizar_espacios(antes)
        registrar_transformacion(tabla, columna, "Normalización de espacios", antes, df[columna])

email_antes = clientes_limpios["email"].copy()
clientes_limpios["email"] = normalizar_espacios(email_antes).str.lower()
registrar_transformacion(
    "clientes", "email", "Espacios y conversión a minúsculas",
    email_antes, clientes_limpios["email"],
)

bitacora_transformaciones_df = pd.DataFrame(bitacora_transformaciones)
display(bitacora_transformaciones_df.query("registros_modificados > 0"))


,tabla,columna,proceso,registros_modificados
7,ventas,sucursal,Unificación de sucursales,8
8,stock,sucursal,Unificación de sucursales,6
9,productos,categoria,Unificación de categorías,5
10,clientes,ciudad,Unificación de ciudades,7


In [9]:
categorias_validas = {"Alimentos", "Snacks", "Higiene", "Accesorios", "Salud"}
sucursales_ventas_validas = {"Centro", "City Bell", "Los Hornos", "Ecommerce"}
sucursales_stock_validas = {"Centro", "City Bell", "Los Hornos"}

assert set(productos_limpios["categoria"].dropna()).issubset(categorias_validas)
assert set(ventas_limpias["sucursal"].dropna()).issubset(sucursales_ventas_validas)
assert set(stock_limpio["sucursal"].dropna()).issubset(sucursales_stock_validas)

print("✓ Textos y categorías normalizados correctamente.")


✓ Textos y categorías normalizados correctamente.


## 5. Conversión de fechas e importes


In [10]:
def convertir_fecha(serie):
    return pd.to_datetime(
        serie,
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )


ventas_limpias["fecha_original"] = ventas_limpias["fecha"].copy()
clientes_limpios["fecha_alta_original"] = clientes_limpios["fecha_alta"].copy()
stock_limpio["fecha_original"] = stock_limpio["fecha"].copy()

ventas_limpias["fecha"] = convertir_fecha(ventas_limpias["fecha_original"])
clientes_limpios["fecha_alta"] = convertir_fecha(clientes_limpios["fecha_alta_original"])
stock_limpio["fecha"] = convertir_fecha(stock_limpio["fecha_original"])

inicio_ventas = pd.Timestamp("2025-01-01")
fin_datos = pd.Timestamp("2026-08-31")

ventas_limpias["fecha_invalida"] = ventas_limpias["fecha"].isna()
ventas_limpias["fecha_fuera_rango"] = (
    ventas_limpias["fecha"].notna()
    & ~ventas_limpias["fecha"].between(inicio_ventas, fin_datos)
)

clientes_limpios["fecha_alta_invalida"] = (
    clientes_limpios["fecha_alta"].isna()
    & clientes_limpios["fecha_alta_original"].notna()
)
clientes_limpios["fecha_alta_fuera_rango"] = (
    clientes_limpios["fecha_alta"].notna()
    & (clientes_limpios["fecha_alta"] > fin_datos)
)

stock_limpio["fecha_invalida"] = stock_limpio["fecha"].isna()
stock_limpio["fecha_fuera_rango"] = (
    stock_limpio["fecha"].notna()
    & (stock_limpio["fecha"] != fin_datos)
)

print("✓ Fechas convertidas y validadas.")


✓ Fechas convertidas y validadas.


In [11]:
def convertir_importe_argentino(serie):
    def convertir_valor(valor):
        if pd.isna(valor):
            return np.nan
        if isinstance(valor, (int, float, np.number)):
            return float(valor)

        texto = (
            str(valor).strip()
            .replace("$", "")
            .replace(" ", "")
            .replace(".", "")
            .replace(",", ".")
        )
        return pd.to_numeric(texto, errors="coerce")

    return serie.apply(convertir_valor)


for columna in ["cantidad", "precio_unitario", "descuento"]:
    ventas_limpias[columna] = pd.to_numeric(ventas_limpias[columna], errors="coerce")

for columna in ["stock_actual", "stock_minimo"]:
    stock_limpio[columna] = pd.to_numeric(stock_limpio[columna], errors="coerce")

productos_limpios["costo_original"] = productos_limpios["costo"].copy()
productos_limpios["precio_lista_original"] = productos_limpios["precio_lista"].copy()
productos_limpios["costo"] = convertir_importe_argentino(productos_limpios["costo_original"])
productos_limpios["precio_lista"] = convertir_importe_argentino(productos_limpios["precio_lista_original"])

display(productos_limpios[["costo", "precio_lista"]].describe())


,costo,precio_lista
count,195.000000,200.000000
mean,18907.841026,28567.500000
std,14247.648765,21034.606416
min,1900.000000,2700.000000
25%,6500.000000,9600.000000
50%,14500.000000,22050.000000
75%,30050.000000,46200.000000
max,58495.000000,84400.000000


## 6. Reglas de calidad y separación de incidencias


In [12]:
productos_limpios["costo_faltante"] = productos_limpios["costo"].isna()
productos_limpios["precio_invalido"] = (
    productos_limpios["precio_lista"].isna()
    | (productos_limpios["precio_lista"] <= 0)
)
productos_limpios["costo_invalido"] = (
    productos_limpios["costo"].notna()
    & (productos_limpios["costo"] <= 0)
)
productos_limpios["costo_mayor_precio"] = (
    productos_limpios["costo"].notna()
    & productos_limpios["precio_lista"].notna()
    & (productos_limpios["costo"] > productos_limpios["precio_lista"])
)
productos_limpios["requiere_revision"] = productos_limpios[[
    "costo_faltante", "precio_invalido", "costo_invalido", "costo_mayor_precio"
]].any(axis=1)

email_informado = clientes_limpios["email"].notna()
email_valido = clientes_limpios["email"].str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+",
    na=False,
)
clientes_limpios["email_invalido"] = email_informado & ~email_valido
clientes_limpios["ciudad_faltante"] = clientes_limpios["ciudad"].isna()
clientes_limpios["requiere_revision"] = clientes_limpios[[
    "fecha_alta_invalida", "fecha_alta_fuera_rango", "email_invalido", "ciudad_faltante"
]].any(axis=1)

productos_observados = productos_limpios[productos_limpios["requiere_revision"]].copy()
clientes_observados = clientes_limpios[clientes_limpios["requiere_revision"]].copy()

print("Productos para revisar:", len(productos_observados))
print("Clientes para revisar:", len(clientes_observados))


Productos para revisar: 9
Clientes para revisar: 49


In [13]:
ventas_limpias["error_cantidad"] = (
    ventas_limpias["cantidad"].isna()
    | (ventas_limpias["cantidad"] <= 0)
)
ventas_limpias["cantidad_atipica"] = ventas_limpias["cantidad"] > 50
ventas_limpias["error_precio"] = (
    ventas_limpias["precio_unitario"].isna()
    | (ventas_limpias["precio_unitario"] <= 0)
)
ventas_limpias["error_descuento"] = (
    ventas_limpias["descuento"].isna()
    | ~ventas_limpias["descuento"].between(0, 1)
)

columnas_error_ventas = [
    "fecha_invalida",
    "fecha_fuera_rango",
    "error_cantidad",
    "cantidad_atipica",
    "error_precio",
    "error_descuento",
]
ventas_limpias["requiere_revision"] = ventas_limpias[columnas_error_ventas].any(axis=1)


def motivos_venta(fila):
    reglas = [
        ("fecha_invalida", "Fecha inválida"),
        ("fecha_fuera_rango", "Fecha fuera de rango"),
        ("error_cantidad", "Cantidad inválida"),
        ("cantidad_atipica", "Cantidad atípica"),
        ("error_precio", "Precio inválido"),
        ("error_descuento", "Descuento inválido"),
    ]
    return ", ".join(texto for columna, texto in reglas if fila[columna])


ventas_limpias["motivo_revision"] = ventas_limpias.apply(motivos_venta, axis=1)
ventas_prevalidas = ventas_limpias[~ventas_limpias["requiere_revision"]].copy().reset_index(drop=True)
ventas_revision = ventas_limpias[ventas_limpias["requiere_revision"]].copy().reset_index(drop=True)

assert len(ventas_prevalidas) + len(ventas_revision) == len(ventas_limpias)
display(ventas_limpias[columnas_error_ventas].sum().rename("cantidad").to_frame())


,cantidad
fecha_invalida,3
fecha_fuera_rango,0
error_cantidad,0
cantidad_atipica,4
error_precio,12
error_descuento,0


In [14]:
stock_limpio["stock_actual_invalido"] = (
    stock_limpio["stock_actual"].isna()
    | (stock_limpio["stock_actual"] < 0)
)
stock_limpio["stock_minimo_invalido"] = (
    stock_limpio["stock_minimo"].isna()
    | (stock_limpio["stock_minimo"] < 0)
)

columnas_error_stock = [
    "fecha_invalida",
    "fecha_fuera_rango",
    "stock_actual_invalido",
    "stock_minimo_invalido",
]
stock_limpio["requiere_revision"] = stock_limpio[columnas_error_stock].any(axis=1)


def motivos_stock(fila):
    reglas = [
        ("fecha_invalida", "Fecha inválida"),
        ("fecha_fuera_rango", "Fecha fuera de rango"),
        ("stock_actual_invalido", "Stock actual inválido"),
        ("stock_minimo_invalido", "Stock mínimo inválido"),
    ]
    return ", ".join(texto for columna, texto in reglas if fila[columna])


stock_limpio["motivo_revision"] = stock_limpio.apply(motivos_stock, axis=1)
stock_prevalido = stock_limpio[~stock_limpio["requiere_revision"]].copy().reset_index(drop=True)
stock_revision = stock_limpio[stock_limpio["requiere_revision"]].copy().reset_index(drop=True)

assert len(stock_prevalido) + len(stock_revision) == len(stock_limpio)
display(stock_limpio[columnas_error_stock].sum().rename("cantidad").to_frame())


,cantidad
fecha_invalida,0
fecha_fuera_rango,0
stock_actual_invalido,6
stock_minimo_invalido,6


## 7. Integridad referencial

- Las ventas sin producto válido se separan porque no pueden enriquecerse con categoría o costo.
- Las ventas sin cliente se conservan como `C_INVITADO`.
- Las ventas con un cliente informado que no existe se conservan como `C_NO_ENCONTRADO`.


In [15]:
assert productos_limpios["id_producto"].is_unique
assert clientes_limpios["id_cliente"].is_unique

ids_productos = set(productos_limpios["id_producto"].dropna())
ids_clientes = set(clientes_limpios["id_cliente"].dropna())

ventas_prevalidas["producto_existe"] = ventas_prevalidas["id_producto"].isin(ids_productos)
ventas_revision_integridad = ventas_prevalidas[~ventas_prevalidas["producto_existe"]].copy().reset_index(drop=True)
ventas_integras = ventas_prevalidas[ventas_prevalidas["producto_existe"]].copy().reset_index(drop=True)
ventas_revision_integridad["motivo_revision"] = "Producto inexistente en el maestro"

cliente_texto = ventas_integras["id_cliente"].fillna("").astype("string").str.strip()
ventas_integras["cliente_informado"] = cliente_texto.ne("")
ventas_integras["cliente_existe"] = ventas_integras["id_cliente"].isin(ids_clientes)
ventas_integras["estado_cliente"] = np.select(
    [
        ~ventas_integras["cliente_informado"],
        ventas_integras["cliente_existe"],
    ],
    ["Invitado", "Registrado"],
    default="No encontrado",
)
ventas_integras["id_cliente_modelo"] = ventas_integras["id_cliente"].copy()
ventas_integras.loc[ventas_integras["estado_cliente"] == "Invitado", "id_cliente_modelo"] = "C_INVITADO"
ventas_integras.loc[ventas_integras["estado_cliente"] == "No encontrado", "id_cliente_modelo"] = "C_NO_ENCONTRADO"

assert len(ventas_integras) + len(ventas_revision_integridad) == len(ventas_prevalidas)
display(ventas_integras["estado_cliente"].value_counts().rename_axis("estado_cliente").reset_index(name="cantidad_lineas"))


,estado_cliente,cantidad_lineas
0,Registrado,17203
1,Invitado,2750
2,No encontrado,13


In [16]:
stock_prevalido["producto_existe"] = stock_prevalido["id_producto"].isin(ids_productos)
stock_revision_integridad = stock_prevalido[~stock_prevalido["producto_existe"]].copy().reset_index(drop=True)
stock_validado = stock_prevalido[stock_prevalido["producto_existe"]].copy().reset_index(drop=True)
stock_revision_integridad["motivo_revision"] = "Producto inexistente en el maestro"

assert len(stock_validado) + len(stock_revision_integridad) == len(stock_prevalido)
print("Stock con producto válido:", len(stock_validado))
print("Stock sin producto válido:", len(stock_revision_integridad))


Stock con producto válido: 585
Stock sin producto válido: 3


## 8. Dimensiones y tablas analíticas


In [17]:
productos_dimension = productos_limpios[[
    "id_producto", "producto", "categoria", "subcategoria",
    "marca", "costo", "precio_lista", "proveedor",
]].copy()

clientes_dimension = clientes_limpios[[
    "id_cliente", "nombre", "ciudad", "fecha_alta", "mascota", "email",
]].copy()

clientes_especiales = pd.DataFrame({
    "id_cliente": ["C_INVITADO", "C_NO_ENCONTRADO"],
    "nombre": ["Cliente invitado", "Cliente no encontrado"],
})
clientes_dimension = pd.concat(
    [clientes_dimension, clientes_especiales],
    ignore_index=True,
)

assert productos_dimension["id_producto"].is_unique
assert clientes_dimension["id_cliente"].is_unique
print("✓ Dimensiones creadas.")


✓ Dimensiones creadas.


In [18]:
ventas_enriquecidas = ventas_integras.merge(
    productos_dimension,
    on="id_producto",
    how="left",
    validate="many_to_one",
    indicator="union_producto",
)
assert ventas_enriquecidas["union_producto"].eq("both").all()
ventas_enriquecidas.drop(columns="union_producto", inplace=True)

clientes_para_merge = clientes_dimension.rename(
    columns={"id_cliente": "id_cliente_modelo"}
)
ventas_enriquecidas = ventas_enriquecidas.merge(
    clientes_para_merge,
    on="id_cliente_modelo",
    how="left",
    validate="many_to_one",
    indicator="union_cliente",
)
assert ventas_enriquecidas["union_cliente"].eq("both").all()
ventas_enriquecidas.drop(columns="union_cliente", inplace=True)

stock_enriquecido = stock_validado.merge(
    productos_dimension,
    on="id_producto",
    how="left",
    validate="many_to_one",
    indicator="union_producto",
)
assert stock_enriquecido["union_producto"].eq("both").all()
stock_enriquecido.drop(columns="union_producto", inplace=True)

print("✓ Ventas y stock enriquecidos.")


✓ Ventas y stock enriquecidos.


## 9. Consolidación de incidencias y controles finales


In [19]:
ventas_observadas = pd.concat(
    [ventas_revision, ventas_revision_integridad],
    ignore_index=True,
    sort=False,
)
stock_observado = pd.concat(
    [stock_revision, stock_revision_integridad],
    ignore_index=True,
    sort=False,
)

resumen_resultado = pd.DataFrame([
    {"tabla": "ventas", "registros_limpios": len(ventas_enriquecidas), "registros_observados": len(ventas_observadas)},
    {"tabla": "stock", "registros_limpios": len(stock_enriquecido), "registros_observados": len(stock_observado)},
    {"tabla": "productos", "registros_limpios": len(productos_dimension), "registros_observados": len(productos_observados)},
    {"tabla": "clientes", "registros_limpios": len(clientes_dimension) - 2, "registros_observados": len(clientes_observados)},
])

assert ventas_enriquecidas["id_linea_venta"].is_unique
assert ventas_enriquecidas["id_producto"].isin(productos_dimension["id_producto"]).all()
assert ventas_enriquecidas["id_cliente_modelo"].isin(clientes_dimension["id_cliente"]).all()
assert stock_enriquecido["id_producto"].isin(productos_dimension["id_producto"]).all()
assert len(ventas_enriquecidas) + len(ventas_observadas) == len(ventas_limpias)
assert len(stock_enriquecido) + len(stock_observado) == len(stock_limpio)

display(resumen_resultado)
print("✓ Notebook ejecutado correctamente de principio a fin.")


,tabla,registros_limpios,registros_observados
0,ventas,19966,34
1,stock,585,15
2,productos,200,9
3,clientes,2000,49


✓ Notebook ejecutado correctamente de principio a fin.


## Metricas comerciales 

In [20]:
ventas_analiticas = ventas_enriquecidas.copy()

In [21]:
ventas_analiticas["venta_bruta"] = (
    ventas_analiticas["cantidad"]
    * ventas_analiticas["precio_unitario"]
)

In [ ]:
#DescuentoAplicado=VentaBruta×Descuento
ventas_analiticas["descuento_aplicado"] = (
    ventas_analiticas["venta_bruta"]
    * ventas_analiticas["descuento"]
)

In [24]:
#FacturacionNeta=VentaBruta−DescuentoAplicado
ventas_analiticas["facturacion_neta"] = (
    ventas_analiticas["venta_bruta"]
    - ventas_analiticas["descuento_aplicado"]
)

In [ ]:
#CostoTotal=Cantidad×CostoUnitario
ventas_analiticas["costo_disponible"] = (
    ventas_analiticas["costo"].notna()
)

ventas_analiticas["costo_total"] = (
    ventas_analiticas["cantidad"]
    * ventas_analiticas["costo"]
)

In [27]:
ventas_analiticas["margen"] = (
    ventas_analiticas["facturacion_neta"]
    - ventas_analiticas["costo_total"]
)

ventas_analiticas["margen_porcentaje"] = np.where(
    ventas_analiticas["facturacion_neta"] > 0,
    (
        ventas_analiticas["margen"]
        / ventas_analiticas["facturacion_neta"]
    ),
    np.nan
)

In [29]:
#Revisamos resultados
display(
    ventas_analiticas[
        [
            "id_linea_venta",
            "id_venta",
            "producto",
            "cantidad",
            "precio_unitario",
            "descuento",
            "venta_bruta",
            "descuento_aplicado",
            "facturacion_neta",
            "costo_total",
            "margen",
            "margen_porcentaje"
        ]
    ].head(10)
)

,id_linea_venta,id_venta,producto,cantidad,precio_unitario,descuento,venta_bruta,descuento_aplicado,facturacion_neta,costo_total,margen,margen_porcentaje
0,LV000001,V000001,Royal Canin Juguete Gato M,1,55310,0.20,55310,11062.0,44248.0,58495.0,-14247.0,-0.321981
1,LV000002,V000002,"König Cachorro Gato 7,5 kg",1,6310,0.00,6310,0.0,6310.0,4500.0,1810.0,0.286846
2,LV000003,V000002,Catit Piel sensible Perro 1 kg,1,8930,0.00,8930,0.0,8930.0,4800.0,4130.0,0.462486
3,LV000004,V000003,Excellent Premium Perro 10 kg,1,24130,0.00,24130,0.0,24130.0,13800.0,10330.0,0.428098
4,LV000005,V000003,Dogit Cepillo dental Perro 2 kg,1,11040,0.05,11040,552.0,10488.0,8400.0,2088.0,0.199085
5,LV000006,V000004,Pedigree Piel sensible Perro 3 kg,2,31880,0.00,63760,0.0,63760.0,48200.0,15560.0,0.244040
6,LV000007,V000004,Catit Tiras Gato 500 g,1,9500,0.00,9500,0.0,9500.0,5500.0,4000.0,0.421053
7,LV000008,V000004,Balanced Piel sensible Gato 15 kg,2,46550,0.15,93100,13965.0,79135.0,64000.0,15135.0,0.191255
8,LV000009,V000004,Kongo Shampoo General 250 ml,2,13780,0.00,27560,0.0,27560.0,NaN,NaN,NaN
9,LV000010,V000005,Royal Canin Pipeta Gato Grande,1,7270,0.00,7270,0.0,7270.0,5600.0,1670.0,0.229711


In [30]:
assert (ventas_analiticas["venta_bruta"] > 0).all()
assert (ventas_analiticas["descuento_aplicado"] >= 0).all()
assert (ventas_analiticas["facturacion_neta"] > 0).all()

print("✓ No existen importes inválidos.")

✓ No existen importes inválidos.


## Cobertura de costos

In [32]:
facturacion_total = (
    ventas_analiticas["facturacion_neta"].sum()
)

facturacion_con_costo = (
    ventas_analiticas.loc[
        ventas_analiticas["costo_disponible"],
        "facturacion_neta"
    ].sum()
)

cobertura_costos = (
    facturacion_con_costo
    / facturacion_total
)

In [33]:
print(
    f"Facturación total: ${facturacion_total:,.2f}"
)

print(
    f"Facturación con costo conocido: "
    f"${facturacion_con_costo:,.2f}"
)

print(
    f"Cobertura de costos: {cobertura_costos:.2%}"
)

Facturación total: $797,723,435.00
Facturación con costo conocido: $772,302,294.50
Cobertura de costos: 96.81%


## Indicadores principales

In [34]:
cantidad_ventas = (
    ventas_analiticas["id_venta"].nunique()
)

In [35]:
unidades_vendidas = (
    ventas_analiticas["cantidad"].sum()
)

In [38]:
descuentos_totales = (
    ventas_analiticas["descuento_aplicado"].sum()
)

In [39]:
ventas_con_costo = ventas_analiticas[
    ventas_analiticas["costo_disponible"]
].copy()

assert np.allclose(
    ventas_con_costo["facturacion_neta"]
    - ventas_con_costo["costo_total"],
    ventas_con_costo["margen"]
)

print("✓ Márgenes validados.")

✓ Márgenes validados.


In [40]:
costo_total_conocido = (
    ventas_con_costo["costo_total"].sum()
)

margen_total_conocido = (
    ventas_con_costo["margen"].sum()
)

margen_porcentaje_conocido = (
    margen_total_conocido
    / facturacion_con_costo
)

In [41]:
tickets = (
    ventas_analiticas
    .groupby("id_venta", as_index=False)
    .agg(
        fecha=("fecha", "min"),
        facturacion_ticket=("facturacion_neta", "sum"),
        unidades_ticket=("cantidad", "sum"),
        productos_diferentes=("id_producto", "nunique")
    )
)

In [42]:
ticket_promedio = (
    tickets["facturacion_ticket"].mean()
)

unidades_promedio_ticket = (
    tickets["unidades_ticket"].mean()
)

## Clientes unicos

In [43]:
clientes_registrados_unicos = (
    ventas_analiticas.loc[
        ventas_analiticas["estado_cliente"]
        == "Registrado",
        "id_cliente_modelo"
    ].nunique()
)

## KPIs

In [44]:
resumen_kpis = pd.DataFrame([
    {
        "indicador": "Facturación neta",
        "valor": facturacion_total
    },
    {
        "indicador": "Ventas",
        "valor": cantidad_ventas
    },
    {
        "indicador": "Unidades vendidas",
        "valor": unidades_vendidas
    },
    {
        "indicador": "Ticket promedio",
        "valor": ticket_promedio
    },
    {
        "indicador": "Descuentos otorgados",
        "valor": descuentos_totales
    },
    {
        "indicador": "Costo total conocido",
        "valor": costo_total_conocido
    },
    {
        "indicador": "Margen conocido",
        "valor": margen_total_conocido
    },
    {
        "indicador": "Margen porcentual conocido",
        "valor": margen_porcentaje_conocido
    },
    {
        "indicador": "Cobertura de costos",
        "valor": cobertura_costos
    },
    {
        "indicador": "Clientes registrados únicos",
        "valor": clientes_registrados_unicos
    }
])

display(resumen_kpis)

,indicador,valor
0,Facturación neta,7.977234e+08
1,Ventas,7.967000e+03
2,Unidades vendidas,2.874000e+04
3,Ticket promedio,1.001285e+05
4,Descuentos otorgados,3.065008e+07
5,Costo total conocido,5.286017e+08
6,Margen conocido,2.437006e+08
7,Margen porcentual conocido,3.155508e-01
8,Cobertura de costos,9.681329e-01
9,Clientes registrados únicos,1.931000e+03


## Evolucion mensual

In [45]:
ventas_analiticas["mes"] = (
    ventas_analiticas["fecha"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

resumen_mensual = (
    ventas_analiticas
    .groupby("mes", as_index=False)
    .agg(
        facturacion_neta=("facturacion_neta", "sum"),
        ventas=("id_venta", "nunique"),
        unidades=("cantidad", "sum"),
        clientes=("id_cliente_modelo", "nunique")
    )
    .sort_values("mes")
)

display(resumen_mensual)

resumen_mensual["ticket_promedio"] = (
    resumen_mensual["facturacion_neta"]
    / resumen_mensual["ventas"]
)

,mes,facturacion_neta,ventas,unidades,clientes
0,2025-01-01,41234991.0,401,1473,327
1,2025-02-01,37808683.5,381,1340,304
2,2025-03-01,38656443.5,396,1385,315
3,2025-04-01,42224239.0,406,1497,331
4,2025-05-01,39185323.0,410,1469,324
5,2025-06-01,38562140.0,367,1332,303
6,2025-07-01,41144437.5,415,1502,334
7,2025-08-01,40250046.0,396,1421,311
8,2025-09-01,38295270.5,394,1323,320
9,2025-10-01,42151886.5,405,1545,324


## Resumen por sucursal

In [46]:
resumen_sucursal = (
    ventas_analiticas
    .groupby("sucursal", as_index=False)
    .agg(
        facturacion_neta=("facturacion_neta", "sum"),
        ventas=("id_venta", "nunique"),
        unidades=("cantidad", "sum"),
        clientes=("id_cliente_modelo", "nunique")
    )
)

resumen_sucursal["ticket_promedio"] = (
    resumen_sucursal["facturacion_neta"]
    / resumen_sucursal["ventas"]
)

resumen_sucursal = resumen_sucursal.sort_values(
    "facturacion_neta",
    ascending=False
)

display(resumen_sucursal)

,sucursal,facturacion_neta,ventas,unidades,clientes,ticket_promedio
0,Centro,283111836.5,2805,10272,1424,100931.136007
2,Ecommerce,195335090.0,1941,6944,1154,100636.316332
1,City Bell,184853587.5,1888,6771,1124,97909.739142
3,Los Hornos,134422921.0,1337,4753,873,100540.703815


## Resumen por categoria

In [47]:
ventas_analiticas["facturacion_con_costo"] = (
    ventas_analiticas["facturacion_neta"]
    .where(ventas_analiticas["costo_disponible"])
)

In [48]:
resumen_categoria = (
    ventas_analiticas
    .groupby("categoria", as_index=False)
    .agg(
        facturacion_neta=("facturacion_neta", "sum"),
        facturacion_con_costo=(
            "facturacion_con_costo",
            lambda serie: serie.sum(min_count=1)
        ),
        costo_total=(
            "costo_total",
            lambda serie: serie.sum(min_count=1)
        ),
        margen=(
            "margen",
            lambda serie: serie.sum(min_count=1)
        ),
        ventas=("id_venta", "nunique"),
        unidades=("cantidad", "sum")
    )
)

In [49]:
resumen_categoria["margen_porcentaje"] = (
    resumen_categoria["margen"]
    / resumen_categoria["facturacion_con_costo"]
)

resumen_categoria = resumen_categoria.sort_values(
    "facturacion_neta",
    ascending=False
)

display(resumen_categoria)

,categoria,facturacion_neta,facturacion_con_costo,costo_total,margen,ventas,unidades,margen_porcentaje
1,Alimentos,528315715.5,508194555.5,348684234.0,159510321.5,6610,16795,0.313876
0,Accesorios,167825079.5,164461762.0,110652215.0,53809547.0,2356,3762,0.327186
3,Salud,48899999.0,48344863.5,33842300.0,14502563.5,1436,2245,0.299981
2,Higiene,36220450.0,34838922.5,24681960.0,10156962.5,2045,3306,0.291541
4,Snacks,16462191.0,16462191.0,10741000.0,5721191.0,1739,2632,0.347535


## Exporto datos

In [52]:
# Exportar datos procesados

RUTA_PROCESADOS = RUTA_PROYECTO / "datos" / "procesados"
RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)

ventas_enriquecidas.to_csv(
    RUTA_PROCESADOS / "ventas_enriquecidas.csv",
    index=False,
    encoding="utf-8-sig"
)

stock_enriquecido.to_csv(
    RUTA_PROCESADOS / "stock_enriquecido.csv",
    index=False,
    encoding="utf-8-sig"
)

productos_dimension.to_csv(
    RUTA_PROCESADOS / "productos_dimension.csv",
    index=False,
    encoding="utf-8-sig"
)

clientes_dimension.to_csv(
    RUTA_PROCESADOS / "clientes_dimension.csv",
    index=False,
    encoding="utf-8-sig"
)

ventas_observadas.to_csv(
    RUTA_PROCESADOS / "ventas_observadas.csv",
    index=False,
    encoding="utf-8-sig"
)

stock_observado.to_csv(
    RUTA_PROCESADOS / "stock_observado.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Archivos exportados en: {RUTA_PROCESADOS}")

Archivos exportados en: /home/ips/Repositorio/Análisis de Datos/huellitas/datos/procesados
